# 05 · Identification at scale — labelled data for ML

`dfxm-identify` inverts the tutorial so far: instead of one known dislocation, it renders *labelled* images across slip systems, characters and positions — training data for identification models ([Borgi 2025](../docs/references.md#borgi-2025)). This is the slim cut; the full walkthrough is [identification_ml_tutorial](identification_ml_tutorial/dfxm_identify_ml_tutorial.ipynb).

In [ ]:
%matplotlib inline
import subprocess
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import PowerNorm

import dfxm_geo.direct_space.forward_model as fm

HERE = Path.cwd()
assert HERE.name == "examples", "Run this notebook from the examples/ folder"
IMG, OUT, KERNEL_DIR = HERE / "img", HERE / "out_05", HERE / "kernel"
for d in (IMG, OUT, KERNEL_DIR):
    d.mkdir(exist_ok=True)
fm.pkl_fpath = KERNEL_DIR.resolve().as_posix() + "/"

KERNEL_FILE = KERNEL_DIR / "Resq_i_h-1_k1_l-1_17keV_tutorial.npz"
if not KERNEL_FILE.exists():
    boot = OUT / "bootstrap.toml"
    boot.write_text(
        "[reciprocal]\nNrays = 2_000_000\nseed = 0\nbeamstop = false\n", encoding="utf-8"
    )
    subprocess.run(
        [
            sys.executable,
            "-c",
            "from dfxm_geo.reciprocal_space.kernel import cli_main; cli_main()",
            "--config",
            str(boot),
            "--output",
            str(KERNEL_FILE),
        ],
        check=True,
    )

# Subprocesses don't inherit fm.pkl_fpath - prefix it into their bootstrap.
KERNEL_ENV = "import dfxm_geo.direct_space.forward_model as fm; fm.pkl_fpath = %r; " % (
    KERNEL_DIR.resolve().as_posix() + "/"
)

## A tiny labelled dataset

`mode = "multi"` drops two random dislocations per sample, renders the noisy combined image plus a noiseless per-dislocation render each (`render_per_dislocation`), and stores the labels (slip plane, Burgers vector, rotation, position) next to the images. Four samples are enough to see the shape of the data.

In [ ]:
ID_TOML = """
mode = "multi"

[reciprocal]
hkl = [-1, 1, -1]
keV = 17.0

[scan.phi]
value = 1.25e-4

[noise]
poisson_noise = true
rng_seed = 3
intensity_scale = 7.0

[multi]
n_samples = 4
pos_std_um = 5.0
render_per_dislocation = true

[io]
include_perfect_crystal = false
write_strain_provenance = false
"""
cfg_file = OUT / "identify_multi.toml"
cfg_file.write_text(ID_TOML, encoding="utf-8")
master = OUT / "identify" / "dfxm_identify.h5"
if not master.exists():
    subprocess.run(
        [
            sys.executable,
            "-c",
            KERNEL_ENV + "from dfxm_geo.pipeline import cli_main_identify; cli_main_identify()",
            "--config",
            str(cfg_file),
            "--output",
            str(OUT / "identify"),
            "--mode",
            "multi",
        ],
        check=True,
    )
print("master:", master)

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
with h5py.File(master, "r") as f:
    scans = sorted((k for k in f if k.endswith(".1")), key=lambda s: int(s.split(".")[0]))[:4]
    for row, scan in zip(axes, scans, strict=False):
        combined = f[f"/{scan}/instrument/dfxm_sim_detector/data"][0]
        row[0].imshow(combined, cmap="magma", norm=PowerNorm(0.45))
        row[0].set_ylabel(f"scan {scan}", fontsize=8)
        labels = []
        for di in sorted(f[f"/{scan}/sample/dislocations"].keys()):
            g = f[f"/{scan}/sample/dislocations/{di}"]
            n_ = g["slip_plane_normal"][...].astype(int)
            b_ = g["burgers"][...].astype(int)
            labels.append(f"n={n_.tolist()} b={b_.tolist()} α={float(g['rotation_deg'][()]):.0f}°")
        scan_no = int(scan.split(".")[0])
        for col, di, lab in zip(row[1:], ("dis0", "dis1"), labels, strict=False):
            solo = OUT / "identify" / f"scan{scan_no:04d}" / f"dfxm_sim_detector_{di}_0000.h5"
            with h5py.File(solo, "r") as fs:
                col.imshow(
                    fs["/entry_0000/dfxm_sim_detector/image"][0], cmap="magma", norm=PowerNorm(0.45)
                )
            col.set_title(lab, fontsize=7)
        for ax in row:
            ax.set_xticks([])
            ax.set_yticks([])
fig.suptitle("combined (noisy) | per-dislocation renders with labels")
fig.tight_layout()
fig.savefig(IMG / "05_identification_preview.png", dpi=110, bbox_inches="tight")

## Throughput: how 100k images become practical

Per-config cost is dominated by import/JIT and Hg geometry, so the fan-out launcher (`scripts/fanout.py`) keeps a persistent worker pool, and the v2.5.1 fused kernels cut the geometry stage ~15×; `write_strain_provenance = false` is the storage lever (0.51 TB → ~3 GB per 100k images). The numbers below are the measured rows from [docs/cluster-profiling.md](../docs/cluster-profiling.md).

In [ ]:
runs = ["v2.4.0\nsubprocess", "v2.5.1\nisolate", "v2.5.1\npool"]
laptop = [703, 1309, 2717]  # configs/hour, 8 workers, 16-config manifest
cluster = [2254, 8173, 12993]  # 32-core LSF node; pool row at 16x2 sizing

x = np.arange(3)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, laptop, 0.4, label="laptop (8 workers)")
ax.bar(x + 0.2, cluster, 0.4, label="cluster node (16x2)")
for i, (lv, cv) in enumerate(zip(laptop, cluster, strict=False)):
    ax.text(i - 0.2, lv, str(lv), ha="center", va="bottom", fontsize=8)
    ax.text(i + 0.2, cv, str(cv), ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(runs)
ax.set_ylabel("identify configs / hour")
ax.set_title("Fan-out throughput (docs/cluster-profiling.md; cluster speedup 5.76x)")
ax.legend()
fig.tight_layout()

At the measured 12 993 configs/hour a 100k-image campaign is an overnight job on a handful of nodes. Recipes (LSF arrays, seed sharding, storage budgets): [identification_ml_tutorial](identification_ml_tutorial/dfxm_identify_ml_tutorial.ipynb) §8 and [docs/cluster-runs.md](../docs/cluster-runs.md). All references: [references page](../docs/references.md).